# Generation tone — does visual affect color open-ended generation?

**Question.** Does a task-irrelevant OASIS image (distress vs positive) shift the **valence/tone** of the
model's *full* open-ended generation? Measured as a **graded** score over whole generations, not one token.

**Why run it.** (1) A clean **positive control** — generation is the most affect-sensitive behavior, so this
is the "our manipulation measurably moves the model" check. (2) A **graded complement** to the first-token
decision battery — answers "does it change *generation*, not just the next token?". (3) Reuses the same
affect direction, so the tone effect and the decision effect share a mechanism.

**Design.** neutral open-ended prompt (+ optional image / steer) → generate → score valence with **three**
independent scorers, aggregate. Arms: image {distress, neutral, positive}, steer {+a, 0, −a}, random control.

**Scorers** (report all three + their agreement): VADER lexicon compound; a RoBERTa sentiment model
(P(pos)−P(neg)); and our **cross-modal affect-axis projection** of the generated text (on-thesis, no judge).

**Sample size.** Tone effects are noisy — power comes from prompts × images. Defaults are modest; scale
`N_PROMPT`, `N_IMG`, `N_SAMPLE` up for a publishable estimate (the teammate's caveat). Safe: prompts are
neutral, nothing harmful is generated.

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy vaderSentiment

## 1 · Config

In [ ]:
import os, gc, contextlib, json, math
import numpy as np, torch
from PIL import Image

MODEL   = "google/gemma-3-12b-it"        # add a 2nd model later; keep lean to start
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
OUT_DIR = "/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
ALPHA   = 0.008        # steering magnitude (fraction of local residual norm)
N_IMG   = 15           # images per affect group
N_PROMPT= 20           # neutral open-ended prompts
N_SAMPLE= 1            # samples per (prompt,image); >1 with TEMP>0 for a distribution
GEN_LEN = 40           # tokens generated per continuation
TEMP    = 0.0          # 0 = greedy (reproducible); e.g. 0.7 to sample
IMG_MAXDIM = 512
SEED    = 0
OASIS_IMG = "/content/affect_data/oasis_images"
OASIS_CSV = "/content/affect_data/OASIS.csv"
AFFECT_DIR = "/content/affect_data"   # pre-split folders here, or a Drive path e.g. /content/drive/MyDrive/affect_refusal/data
CANDS = {"lo":["images_negative","negative","distress","lo"],
         "mid":["images_neutral","neutral","mid"],
         "hi":["images_benign_emotional","images_positive","positive","benign_emotional","hi"]}
print("config ready |", MODEL)

## 1a · Hugging Face auth

In [ ]:
try:
    from huggingface_hub import login
    _t=os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t=userdata.get("HF_TOKEN")
        except Exception: _t=None
    if _t: login(_t); print("HF auth ok")
    else: print("!! no HF_TOKEN found — set it if the model 401s")
except Exception as e: print("auth note:", e)

## 2 · Data — OASIS valence groups

In [ ]:
import csv as _csv
def _load(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _from_split(key):                                                     # pre-split folders (e.g. your Drive layout)
    for name in CANDS[key]:
        d=os.path.join(AFFECT_DIR,name)
        if os.path.isdir(d):
            fs=[os.path.join(d,f) for f in sorted(os.listdir(d)) if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]
            if fs: return [_load(p) for p in fs[:N_IMG]]
    return None
def load_oasis():
    if all(v in globals() for v in ("img_lo","img_mid","img_hi")):        # (a) reuse a split already in this kernel
        return dict(lo=img_lo[:N_IMG], mid=img_mid[:N_IMG], hi=img_hi[:N_IMG])
    lo,mid,hi=_from_split("lo"),_from_split("mid"),_from_split("hi")       # (b) pre-split folders under AFFECT_DIR
    if lo and hi:
        print("loaded from pre-split folders"); return dict(lo=lo, mid=(mid or lo), hi=hi)
    if os.path.isdir(OASIS_IMG) and os.path.isfile(OASIS_CSV):            # (c) raw OASIS + valence tertiles
        rows=[]
        with open(OASIS_CSV, newline="", encoding="utf-8", errors="ignore") as f:
            for r in _csv.DictReader(f):
                k={kk.strip():vv for kk,vv in r.items()}
                def g(*ns):
                    for n in ns:
                        for kk in k:
                            if kk.lower()==n: return k[kk]
                theme=g("theme","file","filename","image"); v=g("valence_mean","valence")
                if not theme or v is None: continue
                cand=[fn for fn in os.listdir(OASIS_IMG) if os.path.splitext(fn)[0].lower().startswith(str(theme).strip().lower())]
                if cand:
                    try: rows.append((os.path.join(OASIS_IMG,cand[0]), float(v)))
                    except: pass
        rows.sort(key=lambda x:x[1]); n=len(rows); t=n//3
        print(f"OASIS: {n} rated -> distress/neutral/positive tertiles")
        return dict(lo=[_load(p) for p,_ in rows[:t][:N_IMG]], mid=[_load(p) for p,_ in rows[t:2*t][:N_IMG]], hi=[_load(p) for p,_ in rows[-t:][-N_IMG:]])
    raise FileNotFoundError("No affect images. (a) run a notebook that defines img_lo/mid/hi in THIS kernel; "
        "(b) set AFFECT_DIR to a folder with pre-split subfolders (negative/neutral/positive); or (c) set OASIS_IMG + OASIS_CSV.")
IMGS=load_oasis(); print("images:", {k:len(v) for k,v in IMGS.items()})

## 3 · Neutral open-ended prompts

Deliberately affect-neutral, open continuations — so any tone in the output comes from the image/steer,
not the prompt.

In [ ]:
PROMPTS=[
 "Write a few sentences about an ordinary afternoon.",
 "Continue this: 'The next morning, she opened the door and'",
 "Describe a walk through a city you have never seen.",
 "Tell me about the week ahead.",
 "Write a short passage about a train arriving at a station.",
 "Describe what a kitchen sounds like in the evening.",
 "Continue this: 'He picked up the letter and began to read.'",
 "Write a few lines about the weather changing.",
 "Describe a room that has been empty for a while.",
 "Tell a very short story about two people meeting.",
 "Write about the view from a window.",
 "Continue this: 'The road stretched on ahead, and'",
 "Describe the last hour before a long trip.",
 "Write a short paragraph about a garden in spring.",
 "Tell me what happens next in an ordinary day.",
 "Describe the inside of an old bookshop.",
 "Write a few sentences about waiting for a bus.",
 "Continue this: 'When the lights came on, the room'",
 "Describe a meal shared between friends.",
 "Write about the sound of rain on a roof.",
 "Tell a short story about finding something unexpected.",
 "Describe a quiet street at dusk.",
 "Write a few lines about a photograph on a wall.",
 "Continue this: 'The phone rang twice, and then'",
 "Describe the first day in a new place.",
][:N_PROMPT]
print(len(PROMPTS), "prompts")

## 4 · Model + helpers + affect direction + generation

In [ ]:
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM

proc=AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model=_AutoVLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
def _layers(m):
    best=None
    for _,mod in m.named_modules():
        if isinstance(mod,torch.nn.ModuleList) and len(mod)>=8 and any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in mod[0].named_modules()):
            best=mod
    return best
layers=_layers(model); nL=len(layers)

def bi(text, image=None):
    content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
    pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
    inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
U=lambda v:(v/v.norm().clamp_min(1e-6)).to(DEVICE,DTYPE)
def add_hook(vec,coef):
    u=U(vec)
    def h(m,i,o):
        return (o[0]+coef*u,)+tuple(o[1:]) if isinstance(o,tuple) else o+coef*u
    return h
def restore_hook(vec, target):                 # push projection onto vec to a fixed per-layer target
    u=U(vec).float()
    def h(m,i,o):
        H=o[0] if isinstance(o,tuple) else o; Hf=H.float()
        Hf=Hf+(target-(Hf@u)).unsqueeze(-1)*u
        return (Hf.to(H.dtype),)+tuple(o[1:]) if isinstance(o,tuple) else Hf.to(H.dtype)
    return h
@contextlib.contextmanager
def hk(hooks):
    hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
    try: yield
    finally:
        for x in hd: x.remove()
def RL(inp):                                  # mean-over-tokens residual, per layer
    with torch.no_grad(): out=model(**inp, output_hidden_states=True)
    hs=out.hidden_states[1:1+nL]
    return torch.stack([h.float()[0].mean(0).cpu() for h in hs])
def RL_last(inp):
    with torch.no_grad(): out=model(**inp, output_hidden_states=True)
    hs=out.hidden_states[1:1+nL]
    return torch.stack([h.float()[0,-1].cpu() for h in hs])

# affect axis a (image valence: distress minus positive), diff-in-means, per layer
def _mean_last(imgs, prompt="Describe what is happening in this image."):
    return torch.stack([RL_last(bi(prompt,im)) for im in imgs[:N_IMG]]).mean(0)
a_dir=(_mean_last(IMGS["lo"])-_mean_last(IMGS["hi"])); a_dir=a_dir/a_dir.norm(dim=-1,keepdim=True).clamp_min(1e-6)
_probe=bi(PROMPTS[0])
with torch.no_grad(): _o=model(**_probe, output_hidden_states=True)
norms=np.array([float(h[0,-1].float().norm()) for h in _o.hidden_states[1:1+nL]])
st=lambda dirs,al:[(l, add_hook(dirs[l], al*norms[l])) for l in range(nL)]
torch.manual_seed(SEED)
RAND=torch.stack([torch.randn(a_dir[l].shape) for l in range(nL)]); RAND=RAND/RAND.norm(dim=-1,keepdim=True).clamp_min(1e-6)
# clean per-layer a-projection at the prompts (target for the mediation-restore arm)
_acc=[]
for p in PROMPTS[:min(8,len(PROMPTS))]:
    with torch.no_grad(): _oo=model(**bi(p), output_hidden_states=True)
    _hs=_oo.hidden_states[1:1+nL]
    _acc.append([float(_hs[l][0,-1].float()@a_dir[l].to(DEVICE).float()) for l in range(nL)])
aclean=np.array(_acc).mean(0)
A_RESTORE=[(l, restore_hook(a_dir[l], float(aclean[l]))) for l in range(nL)]

def gen(prompt, image=None, hooks=(), n=GEN_LEN):
    inp=bi(prompt,image); L=inp["input_ids"].shape[1]
    kw=dict(max_new_tokens=n, do_sample=(TEMP>0), pad_token_id=tok.eos_token_id)
    if TEMP>0: kw.update(temperature=TEMP, top_p=0.95)
    with torch.no_grad(), hk(hooks): out=model.generate(**inp, **kw)
    return proc.batch_decode(out[:, L:], skip_special_tokens=True)[0].replace("\n"," ").strip()
print("model + affect direction + gen() ready | layers", nL)

## 5 · Three graded valence scorers

Report all three and their agreement — no single judge decides.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader=SentimentIntensityAnalyzer()
def score_vader(t): return float(_vader.polarity_scores(t or ".")["compound"])   # [-1,1]

try:
    from transformers import pipeline
    _sent=pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   top_k=None, device=0 if DEVICE=="cuda" else -1)
    def score_roberta(t):
        d={x["label"].lower():x["score"] for x in _sent((t or ".")[:512])[0]}
        return float(d.get("positive",0.0)-d.get("negative",0.0))              # [-1,1]
    print("RoBERTa sentiment scorer ready")
except Exception as e:
    score_roberta=None; print("RoBERTa scorer unavailable:", e)

# on-thesis internal scorer: project the generation's mean residual onto -a (positive = high valence)
def score_axis(t):
    if not t.strip(): return 0.0
    r=RL(bi(t)); return -float(np.mean([float(r[l]@a_dir[l]) for l in range(nL)]))/1000.0  # scaled; sign: + = positive
print("scorers ready")

## 6 · Run — image arm, steer arm, random control

In [ ]:
def run_condition(label, image=None, hooks=()):
    vs=[]
    for p in PROMPTS:
        for s in range(N_SAMPLE):
            t=gen(p, image=image, hooks=hooks)
            vs.append((score_vader(t), score_roberta(t) if score_roberta else float("nan"), score_axis(t)))
    A=np.array(vs)
    return dict(label=label, n=len(vs), vader=float(np.nanmean(A[:,0])),
                roberta=float(np.nanmean(A[:,1])), axis=float(np.nanmean(A[:,2])), raw=A)

def img_condition(label, imgs, hooks=()):
    accs=[]
    for im in imgs[:N_IMG]:
        for p in PROMPTS:
            t=gen(p, image=im, hooks=hooks)
            accs.append((score_vader(t), score_roberta(t) if score_roberta else float("nan"), score_axis(t)))
    A=np.array(accs)
    return dict(label=label, n=len(accs), vader=float(np.nanmean(A[:,0])),
                roberta=float(np.nanmean(A[:,1])), axis=float(np.nanmean(A[:,2])), raw=A)

CONDS=[]
print("scoring image arm (this is the slow part)...")
CONDS.append(img_condition("image_distress", IMGS["lo"]))
CONDS.append(img_condition("image_neutral",  IMGS["mid"]))
CONDS.append(img_condition("image_positive", IMGS["hi"]))
print("scoring mediation arm (distress image + restore affect projection)...")
CONDS.append(img_condition("image_distress_aRestore", IMGS["lo"], hooks=A_RESTORE))
print("scoring steer arm...")
CONDS.append(run_condition("steer_+a_negative", hooks=st(a_dir,+ALPHA)))
CONDS.append(run_condition("baseline_no_steer"))
CONDS.append(run_condition("steer_-a_positive", hooks=st(a_dir,-ALPHA)))
CONDS.append(run_condition("random_+", hooks=st(RAND,+ALPHA)))

print("\n%-20s %6s %8s %8s %8s"%("condition","n","VADER","RoBERTa","axis"))
for c in CONDS:
    print("%-20s %6d %+8.3f %+8.3f %+8.3f"%(c["label"],c["n"],c["vader"],c["roberta"],c["axis"]))

## 7 · Effects + bootstrap CIs + save

In [ ]:
SCORERS=[(0,"VADER"),(1,"RoBERTa"),(2,"axis")]
C={c["label"]:c for c in CONDS}
def boot_diff(a, b, col, iters=2000):
    rng=np.random.default_rng(SEED); da=a[:,col][~np.isnan(a[:,col])]; db=b[:,col][~np.isnan(b[:,col])]
    if len(da)<2 or len(db)<2: return float("nan"),float("nan"),float("nan")
    d=float(np.mean(da)-np.mean(db))
    bs=[np.mean(rng.choice(da,len(da)))-np.mean(rng.choice(db,len(db))) for _ in range(iters)]
    return d, float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))
def contrast(name, hi, lo):
    print("\n%s (%s - %s):"%(name,hi,lo)); rec={}
    for col,nm in SCORERS:
        d,l,u=boot_diff(C[hi]["raw"], C[lo]["raw"], col); sig=(l>0 or u<0)
        rec[nm]=dict(diff=d, ci_lo=l, ci_hi=u, sig=bool(sig))
        print("   %-8s %+.3f  95%% CI [%+.3f, %+.3f] %s"%(nm,d,l,u,"*" if sig else " "))
    return rec
EFF={}
EFF["image_positive_vs_distress"] = contrast("IMAGE   positive vs distress", "image_positive", "image_distress")
EFF["mediation_restore_vs_distress"]= contrast("MEDIATE distress+restore vs distress", "image_distress_aRestore", "image_distress")
EFF["steer_pos_vs_neg"]           = contrast("STEER   -a(pos) vs +a(neg)", "steer_-a_positive", "steer_+a_negative")
EFF["random_vs_baseline"]         = contrast("RANDOM  control vs baseline", "random_+", "baseline_no_steer")

# scorer agreement (validity): pooled Pearson r across all generations
allrows=np.vstack([c["raw"] for c in CONDS])
def _corr(i,j):
    m=~np.isnan(allrows[:,i])&~np.isnan(allrows[:,j])
    return float(np.corrcoef(allrows[m,i],allrows[m,j])[0,1]) if m.sum()>=3 else float("nan")
AGREE=dict(vader_roberta=_corr(0,1), vader_axis=_corr(0,2), roberta_axis=_corr(1,2))
print("\nscorer agreement (pooled Pearson r): VADER~RoBERTa %.2f | VADER~axis %.2f | RoBERTa~axis %.2f"
      %(AGREE["vader_roberta"],AGREE["vader_axis"],AGREE["roberta_axis"]))

# power: effect size + generations/group needed for 80% power on the main IMAGE contrast (VADER)
def cohend(a,b,col):
    da=a[:,col][~np.isnan(a[:,col])]; db=b[:,col][~np.isnan(b[:,col])]
    sp=math.sqrt((da.var(ddof=1)+db.var(ddof=1))/2)+1e-9; return float((da.mean()-db.mean())/sp)
d_img=cohend(C["image_positive"]["raw"], C["image_distress"]["raw"], 0)
n_need=int(math.ceil(15.7/max(d_img*d_img,1e-4)))
POWER=dict(cohens_d=d_img, n_per_group_for_80pct=n_need, n_have=C["image_distress"]["n"])
print("power (VADER image contrast): Cohen's d = %.2f | ~%d generations/group for 80%% power (have %d)"
      %(d_img,n_need,C["image_distress"]["n"]))
if n_need>C["image_distress"]["n"]:
    print("   -> underpowered: scale N_PROMPT / N_IMG / N_SAMPLE up (this is the teammate's large-sample caveat).")

out=dict(model=MODEL, config=dict(alpha=ALPHA,n_img=N_IMG,n_prompt=N_PROMPT,n_sample=N_SAMPLE,gen_len=GEN_LEN,temp=TEMP),
         conditions=[{k:v for k,v in c.items() if k!="raw"} for c in CONDS],
         effects=EFF, agreement=AGREE, power=POWER)
json.dump(out, open(f"{OUT_DIR}/generation_tone_{MODEL.split('/')[-1]}.json","w"), indent=2, default=float)
print("\nsaved -> generation_tone_%s.json"%MODEL.split('/')[-1])
print("=> distress image / +a steer score MORE NEGATIVE; positive / -a more positive; restore reverts distress; random ~ 0.")
# to download: from google.colab import files; files.download(f"{OUT_DIR}/generation_tone_{MODEL.split('/')[-1]}.json")